In [0]:

from pyspark.sql.functions import *
from pyspark.sql.window import Window


In [0]:

# -----------------------------------------
# Paths
# -----------------------------------------

silver_sales_path = "/Volumes/workspace/m5_project/silver/silver_sales"

gold_demand_forecast_path = "/Volumes/workspace/m5_project/gold/gold_demand_forecast"


In [0]:
# -----------------------------------------
# Load Silver Sales
# -----------------------------------------

silver_sales_df = spark.read.format("parquet").load(
    silver_sales_path
)

display(silver_sales_df)

id,item_id,dept_id,cat_id,store_id,state_id,day,sales,date,wm_yr_wk
HOUSEHOLD_1_363_TX_3_validation,HOUSEHOLD_1_363,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,0,2011-01-29,11101
HOUSEHOLD_1_364_TX_3_validation,HOUSEHOLD_1_364,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,0,2011-01-29,11101
HOUSEHOLD_1_365_TX_3_validation,HOUSEHOLD_1_365,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,0,2011-01-29,11101
HOUSEHOLD_1_366_TX_3_validation,HOUSEHOLD_1_366,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,0,2011-01-29,11101
HOUSEHOLD_1_367_TX_3_validation,HOUSEHOLD_1_367,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,0,2011-01-29,11101
HOUSEHOLD_1_368_TX_3_validation,HOUSEHOLD_1_368,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,0,2011-01-29,11101
HOUSEHOLD_1_369_TX_3_validation,HOUSEHOLD_1_369,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,0,2011-01-29,11101
HOUSEHOLD_1_370_TX_3_validation,HOUSEHOLD_1_370,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,2,2011-01-29,11101
HOUSEHOLD_1_371_TX_3_validation,HOUSEHOLD_1_371,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,0,2011-01-29,11101
HOUSEHOLD_1_372_TX_3_validation,HOUSEHOLD_1_372,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,6,2011-01-29,11101


In [0]:
# Expected important columns:
# item_id
# store_id
# date
# sales_qty
#
# If your sales column is named differently
# (example: sales, quantity_sold, demand)
# replace sales_qty accordingly.

# -----------------------------------------
# Recent Demand Window
# Last 30 Days Approximation
# -----------------------------------------

# Since M5 often has d_x style dates,
# we use latest available rows rather than true date parsing

window_spec = Window.partitionBy(
    "item_id",
    "store_id",
    "cat_id",
    "dept_id",
    "state_id"
).orderBy(
    col("date").desc()
)


In [0]:
recent_sales_df = (
    silver_sales_df
    .withColumn(
        "row_num",
        row_number().over(window_spec)
    )
    .filter(
        col("row_num") <= 30
    )
)

display(recent_sales_df.limit(20))

id,item_id,dept_id,cat_id,store_id,state_id,day,sales,date,wm_yr_wk,row_num
FOODS_1_002_CA_1_validation,FOODS_1_002,FOODS_1,FOODS,CA_1,CA,d_1913,0,2016-04-24,11613,1
FOODS_1_002_CA_1_validation,FOODS_1_002,FOODS_1,FOODS,CA_1,CA,d_1912,0,2016-04-23,11613,2
FOODS_1_002_CA_1_validation,FOODS_1_002,FOODS_1,FOODS,CA_1,CA,d_1911,0,2016-04-22,11612,3
FOODS_1_002_CA_1_validation,FOODS_1_002,FOODS_1,FOODS,CA_1,CA,d_1910,2,2016-04-21,11612,4
FOODS_1_002_CA_1_validation,FOODS_1_002,FOODS_1,FOODS,CA_1,CA,d_1909,1,2016-04-20,11612,5
FOODS_1_002_CA_1_validation,FOODS_1_002,FOODS_1,FOODS,CA_1,CA,d_1908,0,2016-04-19,11612,6
FOODS_1_002_CA_1_validation,FOODS_1_002,FOODS_1,FOODS,CA_1,CA,d_1907,0,2016-04-18,11612,7
FOODS_1_002_CA_1_validation,FOODS_1_002,FOODS_1,FOODS,CA_1,CA,d_1906,1,2016-04-17,11612,8
FOODS_1_002_CA_1_validation,FOODS_1_002,FOODS_1,FOODS,CA_1,CA,d_1905,3,2016-04-16,11612,9
FOODS_1_002_CA_1_validation,FOODS_1_002,FOODS_1,FOODS,CA_1,CA,d_1904,1,2016-04-15,11611,10


In [0]:
# -----------------------------------------
# Demand Forecast Aggregation
# -----------------------------------------

gold_demand_forecast_df = (
    recent_sales_df
    .groupBy(
        "item_id",
        "store_id",
        "cat_id",
        "dept_id",
        "state_id"
    )
    .agg(
        round(
            avg("sales"),
            2
        ).alias(
            "avg_daily_sales"
        ),

        round(
            avg("sales") * 7,
            2
        ).alias(
            "forecast_next_7_days"
        ),

        round(
            avg("sales") * 30,
            2
        ).alias(
            "forecast_next_30_days"
        ),

        max("sales").alias(
            "peak_recent_sales"
        ),

        stddev("sales").alias(
            "sales_volatility"
        ),

        current_date().alias(
            "forecast_generated_date"
        )
    )
)

In [0]:
# # -----------------------------------------
# # Save Gold Demand Forecast
# # -----------------------------------------

# (
#     gold_demand_forecast_df
#     .write
#     .format("delta")
#     .mode("overwrite")
#     .save(gold_demand_forecast_path)
# )

# print("gold_demand_forecast created successfully")
# print(gold_demand_forecast_path)

In [0]:
gold_demand_forecast_df = (
    gold_demand_forecast_df
    .withColumn(
        "category",
        col("cat_id")
    )
    .withColumn(
        "department",
        col("dept_id")
    )
)


In [0]:
# =====================================
# FORECAST CONFIDENCE
# =====================================

gold_demand_forecast_df = gold_demand_forecast_df.withColumn(
    "forecast_confidence",

    # Dead / inactive product
    when(
        col("avg_daily_sales") == 0,
        "LOW"
    )

    # Very unstable demand → poor forecast confidence
    .when(
        col("sales_volatility") > 5,
        "LOW"
    )

    # Moderate instability
    .when(
        col("sales_volatility") > 2,
        "MEDIUM"
    )

    # Stable demand → strong confidence
    .otherwise(
        "HIGH"
    )
)

In [0]:
# =====================================
# VALIDATION
# =====================================

display(
    gold_demand_forecast_df.select(
        "item_id",
        "category",
        "department",
        "store_id",
        "state_id",
        "avg_daily_sales",
        "forecast_next_7_days",
        "forecast_next_30_days",
        "sales_volatility",
        "forecast_confidence",
        "forecast_generated_date"
    ).limit(20)
)


item_id,category,department,store_id,state_id,avg_daily_sales,forecast_next_7_days,forecast_next_30_days,sales_volatility,forecast_confidence,forecast_generated_date
FOODS_1_002,FOODS,FOODS_1,CA_1,CA,0.4,2.8,12.0,0.7239737088005909,HIGH,2026-04-27
FOODS_1_002,FOODS,FOODS_1,CA_4,CA,0.17,1.17,5.0,0.59209349991676,HIGH,2026-04-27
FOODS_1_003,FOODS,FOODS_1,WI_1,WI,1.07,7.47,32.0,0.9444331755018487,HIGH,2026-04-27
FOODS_1_003,FOODS,FOODS_1,WI_3,WI,0.03,0.23,1.0,0.18257418583505533,HIGH,2026-04-27
FOODS_1_005,FOODS,FOODS_1,TX_1,TX,0.73,5.13,22.0,1.2015315896469556,HIGH,2026-04-27
FOODS_1_006,FOODS,FOODS_1,CA_1,CA,1.7,11.9,51.0,1.600646421142796,HIGH,2026-04-27
FOODS_1_006,FOODS,FOODS_1,TX_1,TX,0.47,3.27,14.0,0.7760791522613608,HIGH,2026-04-27
FOODS_1_006,FOODS,FOODS_1,WI_2,WI,1.27,8.87,38.0,1.529780992514115,HIGH,2026-04-27
FOODS_1_009,FOODS,FOODS_1,CA_4,CA,0.0,0.0,0.0,0.0,LOW,2026-04-27
FOODS_1_010,FOODS,FOODS_1,CA_3,CA,0.73,5.13,22.0,0.8683449709106094,HIGH,2026-04-27


In [0]:
(
    gold_demand_forecast_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_demand_forecast_path)
)

print("Gold Demand Forecast created successfully")
print(gold_demand_forecast_path)

Gold Demand Forecast created successfully
/Volumes/workspace/m5_project/gold/gold_demand_forecast


In [0]:
# -----------------------------------------
# Export Gold Demand Forecast CSV
# -----------------------------------------

gold_demand_forecast_export_path = (
    "/Volumes/workspace/m5_project/gold/exports/"
    "gold_demand_forecast_csv"
)

gold_demand_forecast_df.write.mode("overwrite") \
    .option("header", True) \
    .csv(gold_demand_forecast_export_path)

print("Gold Demand Forecast CSV export complete")
print(gold_demand_forecast_export_path)

Gold Demand Forecast CSV export complete
/Volumes/workspace/m5_project/gold/exports/gold_demand_forecast_csv
